# 6. Analysis

## 6.1 Pipeline performance

The computational research question asks whether the proposed pipeline reliably produces structurally and materially valid space frame designs. Three sub-systems are assessed: the CMA-ES optimiser, the MILP material assignment solver, and the GNN structural proxy.

**CMA-ES convergence.** The optimiser ran for 250 generations at 30 evaluations per generation (7,500 total per run), with the structural penalty weight ω₄ linearly annealed from 0.2 to 0.8 across the run. To characterise repeatability, 10 independent runs were executed on the same stock pool A (524 elements, 421 NS / 103 RS). Best-ever fitness improved monotonically in all runs, with a median final fitness of [**X**] and a run-to-run standard deviation of [**X**]. The best design was found at generation [**X**] on average, and the CMA-ES step size σ contracted to [**X**] by generation [**X**] — indicating focused search concentration without premature convergence. Penalty candidates (minimum reuse fraction violations) accounted for [**X%**] of all evaluated designs.

**MILP efficiency.** At each evaluation the three-stage feasibility filter (member length, axial force, EC5 utilisation) reduced the pool of slot–stock candidate pairs from a maximum of [**N_stock × 120**] to approximately [**X**] feasible pairings per evaluation, leaving roughly [**X**] candidates per member slot. The MILP solver returned Optimal status in all configurations; mean solve time per evaluation was [**X s**].

**GNN structural proxy.** The `TrussEdgeSafetyGNN` was trained on 20,000 Karamba3D-simulated space frame configurations (4-layer GNN, hidden dim 64, 200 epochs). On the 2,000-sample held-out test set it achieved a ROC-AUC of **0.863** and a precision–recall AUC of **0.615**. At the validation-tuned threshold θ = 0.30 (chosen to minimise false negatives for structural safety), recall on unsafe members was **86.6%** with a false-negative rate of **11.4%** — roughly 1 in 9 truly unsafe members predicted as safe. This conservative bias is intentional: the fitness function penalises predicted-unsafe members, so false negatives soften rather than eliminate the structural penalty signal. Applied to the final optimised design the GNN predicted [**X%**] structural feasibility; Karamba3D verification confirmed [**X**] unsafe members, giving a prediction error of [**X**] members ([**X%**] of the 120-member truss).

In [ ]:
# =============================================================================
# 6.1.A  CMA-ES convergence — 10-run batch (GA_A, stock A)
# =============================================================================

import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import config
importlib.reload(config)

_PC  = config.PLOT_COLORS
_EXT = _PC["extra_colors"]

BATCH_STEM = "0_GA_A_BATCH_10_20260520"
batch_dir  = config.GA_DATA_PATH / BATCH_STEM

# ── Load all run histories ────────────────────────────────────────────────────
run_dirs  = sorted([d for d in batch_dir.iterdir() if d.is_dir()])
histories = []
run_meta  = []

for d in run_dirs:
    stem = d.name
    hist = pd.read_csv(d / f"{stem}_history.csv")
    histories.append(hist)
    with open(d / f"{stem}_run_config.json", encoding="utf-8") as f:
        rc = json.load(f)
    with open(d / f"{stem}_best_design.json", encoding="utf-8") as f:
        bd = json.load(f)
    run_num = int(stem.split("_RUN")[1].split("_")[0]) if "_RUN" in stem else len(run_meta) + 1
    run_meta.append({
        "run":     run_num,
        "fitness": bd["fitness"],
        "gen":     bd["generation"],
        "cost":    bd["total_cost"],
        "reuse":   bd["reuse_rate"],
        "n_evals": rc["n_evals"],
    })

df_meta = pd.DataFrame(run_meta).sort_values("run").reset_index(drop=True)

print(f"Batch: {BATCH_STEM}  ({len(histories)} runs)")
print(f"\nPer-run summary:")
print(df_meta.to_string(index=False))
print(f"\nBest fitness:   {df_meta['fitness'].min():.4f}  (run {df_meta.loc[df_meta['fitness'].idxmin(), 'run']})")
print(f"Median fitness: {df_meta['fitness'].median():.4f}")
print(f"Std fitness:    {df_meta['fitness'].std():.4f}")
print(f"Mean gen@best:  {df_meta['gen'].mean():.1f}  ±  {df_meta['gen'].std():.1f}")

# ── Stack into (n_runs × n_gen) matrices ──────────────────────────────────────
n_gen  = min(len(h) for h in histories)
gens   = histories[0]["generation"].values[:n_gen]

bf_mat  = np.stack([h["best_ever"].values[:n_gen]  for h in histories])
sig_mat = np.stack([h["mean_sigma"].values[:n_gen] for h in histories])

bf_med, bf_lo, bf_hi   = np.median(bf_mat, 0),  np.percentile(bf_mat, 10, 0),  np.percentile(bf_mat, 90, 0)
sig_med, sig_lo, sig_hi = np.median(sig_mat, 0), np.percentile(sig_mat, 10, 0), np.percentile(sig_mat, 90, 0)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(
    "CMA-ES convergence  —  10 independent runs  ·  stock A  ·  250 gen / 7,500 evals",
    fontweight="bold", fontsize=11,
)

# Panel 1 — fitness convergence
ax = axes[0]
for row in bf_mat:
    ax.plot(gens, row, color=_PC["neutral"], lw=0.8, alpha=0.6, zorder=1)
ax.fill_between(gens, bf_lo, bf_hi, color=_PC["primary"], alpha=0.15, zorder=2)
ax.plot(gens, bf_med, color=_PC["primary"], lw=2.2, zorder=3, label="Median (best-ever)")
ax.set_xlabel("Generation", fontsize=9)
ax.set_ylabel("Best-ever fitness", fontsize=9)
ax.set_title("Fitness convergence", fontsize=10, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
ax.tick_params(labelsize=8)

# Panel 2 — σ (step size)
ax = axes[1]
for row in sig_mat:
    ax.plot(gens, row, color=_PC["neutral"], lw=0.8, alpha=0.6, zorder=1)
ax.fill_between(gens, sig_lo, sig_hi, color=_EXT["muted_teal"], alpha=0.20, zorder=2)
ax.plot(gens, sig_med, color=_EXT["muted_teal"], lw=2.2, zorder=3, label="Median σ")
ax.set_xlabel("Generation", fontsize=9)
ax.set_ylabel("Mean σ (step size)", fontsize=9)
ax.set_title("CMA-ES step size (σ)", fontsize=10, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
ax.tick_params(labelsize=8)

# Panel 3 — per-run best fitness bar
ax = axes[2]
best_run = df_meta.loc[df_meta["fitness"].idxmin(), "run"]
bar_colors = [_PC["primary"] if r == best_run else _PC["secondary"] for r in df_meta["run"]]
ax.bar(df_meta["run"], df_meta["fitness"].abs(), color=bar_colors, alpha=0.85, edgecolor="white", width=0.7)
ax.set_xlabel("Run", fontsize=9)
ax.set_ylabel("|Best fitness|  (higher = better)", fontsize=9)
ax.set_title("Best fitness per run", fontsize=10, fontweight="bold")
ax.set_xticks(df_meta["run"])
ax.grid(axis="y", linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
ax.tick_params(labelsize=8)
ax.legend(handles=[
    mpatches.Patch(color=_PC["primary"],   label="Best run"),
    mpatches.Patch(color=_PC["secondary"], label="Other runs"),
], fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 6.1.B  GNN structural proxy — training and evaluation performance
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt

import config

_PC  = config.PLOT_COLORS
_EXT = _PC["extra_colors"]

MODEL_PREFIX = "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"
model_dir    = config.SM_EXPORT_PATH / MODEL_PREFIX
report_path  = model_dir / f"{MODEL_PREFIX}_training_report.txt"

# ── Parse training history ────────────────────────────────────────────────────
with open(report_path, encoding="utf-8") as f:
    report_text = f.read()

hist_rows = []
for line in report_text.split("TRAINING HISTORY")[1].splitlines()[2:]:
    parts = line.strip().split(",")
    if len(parts) == 3:
        try:
            hist_rows.append([int(parts[0]), float(parts[1]), float(parts[2])])
        except ValueError:
            pass
df_hist    = pd.DataFrame(hist_rows, columns=["epoch", "train_loss", "val_loss"])
best_epoch = int(df_hist.loc[df_hist["val_loss"].idxmin(), "epoch"])
best_val   = df_hist["val_loss"].min()

print(f"Model  : {MODEL_PREFIX}")
print(f"Epochs : 200  |  best epoch = {best_epoch}  |  best val loss = {best_val:.6f}")
print(f"Data   : 20,000 samples  (16k train / 2k val / 2k test)")
print(f"Labels : 19.6% unsafe  (class-imbalanced)")

# ── Evaluation metrics table ──────────────────────────────────────────────────
df_metrics = pd.DataFrame({
    "Value":   [0.863, 0.615, 0.129, 0.729, 0.417, 0.866, 0.563, 0.456, 0.114],
    "θ":       ["—",   "—",   "—",   0.30,  0.30,  0.30,  0.30,  0.30,  0.30 ],
    "Note":    [
        "Area under ROC curve",
        "Area under precision-recall curve",
        "Calibration score (lower = better)",
        "Overall classification accuracy",
        "Of predicted unsafe: truly unsafe",
        "Of truly unsafe: predicted unsafe",
        "Harmonic mean of precision / recall",
        "Matthews correlation coefficient",
        "Unsafe members missed (predicted safe)",
    ],
}, index=["ROC-AUC", "PR-AUC", "Brier score",
          "Accuracy", "Precision", "Recall (unsafe)", "F1", "MCC",
          "False-negative rate"])
df_metrics.index.name = "Metric"

print()
display(df_metrics)

TP, TN, FP, FN = 42860, 127329, 64290, 5521
total = TP + TN + FP + FN
print(f"\nTest set: {total:,} edge-labels  =  {total // 120:,} graphs × 120 edges")
print(f"  TP={TP:,}  TN={TN:,}  FP={FP:,}  FN={FN:,}")

# ── Figure: training curve + metric bars ──────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("GNN structural proxy — training and evaluation", fontweight="bold", fontsize=11)

ax_l.plot(df_hist["epoch"], df_hist["train_loss"], color=_PC["secondary"], lw=1.5, label="Train loss")
ax_l.plot(df_hist["epoch"], df_hist["val_loss"],   color=_PC["primary"],   lw=2.0, label="Val loss")
ax_l.axvline(best_epoch, color=_PC["accent"], lw=1.2, ls="--", label=f"Best epoch {best_epoch}")
ax_l.set_xlabel("Epoch", fontsize=9)
ax_l.set_ylabel("Weighted BCE loss", fontsize=9)
ax_l.set_title("Training curve  (200 epochs)", fontsize=10, fontweight="bold")
ax_l.legend(fontsize=8)
ax_l.grid(linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
ax_l.tick_params(labelsize=8)

bar_names = ["ROC-AUC", "PR-AUC", "Recall(unsafe)", "Accuracy", "Precision", "F1"]
bar_vals  = [0.863,     0.615,    0.866,              0.729,      0.417,       0.563]
bar_cols  = [_PC["primary"] if v >= 0.70 else _EXT["soft_sage_green"] if v >= 0.50 else _PC["danger"]
             for v in bar_vals]

ax_r.bar(range(len(bar_names)), bar_vals, color=bar_cols, alpha=0.85, edgecolor="white")
ax_r.axhline(0.5, color=_PC["neutral"], lw=1.2, ls="--", alpha=0.8)
ax_r.set_xticks(range(len(bar_names)))
ax_r.set_xticklabels(bar_names, fontsize=8.5)
ax_r.set_ylim(0, 1.08)
ax_r.set_ylabel("Score", fontsize=9)
ax_r.set_title("Evaluation metrics  (θ = 0.30)", fontsize=10, fontweight="bold")
ax_r.grid(axis="y", linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
ax_r.tick_params(labelsize=8)
for i, v in enumerate(bar_vals):
    ax_r.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

## 6.2 Static baseline geometry
Constructs the unoptimised baseline: a static regular-grid space frame using default parameters from `c00_headquarter_params` (5×3 cells, 3 m edge, 1.5 m layer height). The geometry is visualised and exported to `01_geometry_data` for Grasshopper inspection. Pipeline comparison against the GA-optimised design follows in §6.3.

In [ ]:
# =============================================================================
# Build static regular-grid space frame
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from collections import Counter
import importlib
import config
import c00_headquarter_params as c00
import c22_stage_geometry as stage_geometry

importlib.reload(stage_geometry)

# ── Build ──────────────────────────────────────────────────────────────────────
static_geo = stage_geometry.run_geometry_from_design({}, sample_id=0)
df_v_s     = static_geo["df_vertices"]
df_e_s     = static_geo["df_edges"]
df_geo_s   = static_geo["df_geometry_overview"]

# ── Classify each member by the layer of its two endpoints ────────────────────
_vlu_s = (
    df_v_s
    .set_index("vertex_index")[["x", "y", "z", "layer", "attribute"]]
    .to_dict("index")
)

def _vk(v):
    s = str(v)
    return s if s.startswith("v") else f"v{s}"

_mtypes = []
for _, e in df_e_s.iterrows():
    l1 = _vlu_s[_vk(e["V1"])]["layer"]
    l2 = _vlu_s[_vk(e["V2"])]["layer"]
    _mtypes.append(
        "top_chord" if (l1 == l2 == "top")
        else "bot_chord" if (l1 == l2 == "bottom")
        else "web"
    )
df_geo_s["member_type"] = _mtypes
_ct = Counter(_mtypes)

# ── Print statistics ──────────────────────────────────────────────────────────
n_top = int((df_v_s["layer"] == "top").sum())
n_bot = int((df_v_s["layer"] == "bottom").sum())

print(f"Static regular-grid space frame")
print(f"  Grid:    {c00.GRID_CELLS_X} × {c00.GRID_CELLS_Y} cells  |  "
      f"edge = {c00.EDGE_LENGTH} m  |  layer h = {c00.LAYER_HEIGHT} m")
print(f"  Nodes:   {len(df_v_s)}  ({n_top} top,  {n_bot} bottom)")
print(f"  Members: {len(df_e_s)}  "
      f"(top chord: {_ct['top_chord']},  "
      f"bottom chord: {_ct['bot_chord']},  "
      f"web: {_ct['web']})")

print(f"\n  Member lengths  (metres)")
print(f"  {'Type':<16} {'min':>6}  {'mean':>6}  {'max':>6}  {'std':>6}")
print(f"  {'-'*44}")
for mt, lbl in [("top_chord","Top chord"), ("bot_chord","Bot chord"), ("web","Web/diagonal")]:
    s = df_geo_s[df_geo_s["member_type"] == mt]["length_m"]
    print(f"  {lbl:<16} {s.min():>6.3f}  {s.mean():>6.3f}  {s.max():>6.3f}  {s.std():>6.3f}")
all_l = df_geo_s["length_m"]
print(f"  {'All':<16} {all_l.min():>6.3f}  {all_l.mean():>6.3f}  {all_l.max():>6.3f}  {all_l.std():>6.3f}")

top_z = df_v_s[df_v_s["layer"] == "top"]["z"].iloc[0]
bot_z = df_v_s[df_v_s["layer"] == "bottom"]["z"].iloc[0]
print(f"\n  z top layer:    {top_z:.3f} m  (after centring)")
print(f"  z bottom layer: {bot_z:.3f} m  (after centring)")

# ── Colour palette ────────────────────────────────────────────────────────────
_C = dict(
    top_chord="#1f77b4",
    bot_chord="#ff7f0e",
    web      ="#bbbbbb",
    support  ="#d62728",
    load     ="#2ca02c",
    bottom_n ="#9467bd",
)
_MSTYLE = dict(
    top_chord=dict(color=_C["top_chord"], lw=2.0, alpha=0.95),
    bot_chord=dict(color=_C["bot_chord"], lw=2.0, alpha=0.95),
    web      =dict(color=_C["web"],       lw=0.85, alpha=0.45),
)

# ── Pre-build segment arrays, grouped by member type ─────────────────────────
_v1k = df_e_s["V1"].map(lambda v: v if str(v).startswith("v") else f"v{v}")
_v2k = df_e_s["V2"].map(lambda v: v if str(v).startswith("v") else f"v{v}")
_all_segs = np.array([
    [[_vlu_s[v1]["x"], _vlu_s[v1]["y"], _vlu_s[v1]["z"]],
     [_vlu_s[v2]["x"], _vlu_s[v2]["y"], _vlu_s[v2]["z"]]]
    for v1, v2 in zip(_v1k, _v2k)
])  # (n_edges, 2, 3)
_mt_arr = np.array(_mtypes)

# ── Node masks (compute once, reuse per subplot) ──────────────────────────────
_sup_m = df_v_s["attribute"] == "support"
_bot_m = (~_sup_m) & (df_v_s["layer"] == "bottom")
_top_m = (~_sup_m) & (df_v_s["layer"] == "top")

_xs = df_v_s["x"].values
_ys = df_v_s["y"].values
_zs = df_v_s["z"].values

# ── Figure: perspective  +  top view ─────────────────────────────────────────
fig_static, (ax_p, ax_t) = plt.subplots(
    1, 2, figsize=(14, 6),
    subplot_kw={"projection": "3d"},
)
fig_static.suptitle(
    f"Static Regular Space Frame  —  "
    f"{c00.GRID_CELLS_X}×{c00.GRID_CELLS_Y} grid  |  "
    f"edge = {c00.EDGE_LENGTH} m  |  h = {c00.LAYER_HEIGHT} m  |  "
    f"{len(df_v_s)} nodes  ·  {len(df_e_s)} members  "
    f"({_ct['top_chord']} top  /  {_ct['bot_chord']} bot  /  {_ct['web']} webs)",
    fontsize=10.5, fontweight="bold",
)

for ax, elev, azim, title in [
    (ax_p, 26, -50, "Perspective"),
    (ax_t, 85, -90, "Top view  (looking down)"),
]:
    for mt, style in _MSTYLE.items():
        ax.add_collection3d(Line3DCollection(_all_segs[_mt_arr == mt], **style))

    for mask, col, ms in [(_sup_m, _C["support"], 55), (_bot_m, _C["bottom_n"], 28), (_top_m, _C["load"], 22)]:
        sub = df_v_s[mask]
        ax.scatter3D(sub["x"].values, sub["y"].values, sub["z"].values, c=col, s=ms, zorder=5)

    ax.auto_scale_xyz(_xs, _ys, _zs)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("x [m]", fontsize=8)
    ax.set_ylabel("y [m]", fontsize=8)
    ax.set_zlabel("z [m]", fontsize=8, labelpad=10)
    ax.set_box_aspect((1, 0.65, 0.28))
    ax.view_init(elev=elev, azim=azim)
    ax.tick_params(labelsize=7)

_leg = [
    mpatches.Patch(color=_C["top_chord"],
                   label=f"Top chord  ({_ct['top_chord']} members,  3 m each)"),
    mpatches.Patch(color=_C["bot_chord"],
                   label=f"Bottom chord  ({_ct['bot_chord']} members)"),
    mpatches.Patch(color=_C["web"],
                   label=f"Web / diagonal  ({_ct['web']} members)"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["support"], markersize=9,
               label=f"Support node  (4 corners)"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["load"], markersize=7,
               label=f"Top load node  ({n_top - 4})"),
    plt.Line2D([], [], marker="o", color="w",
               markerfacecolor=_C["bottom_n"], markersize=7,
               label=f"Bottom node  ({n_bot})"),
]
ax_p.legend(handles=_leg, loc="upper left", fontsize=7.5,
            framealpha=0.9, bbox_to_anchor=(-0.05, 1.10))

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
# Export static grid to 01_geometry_data for Grasshopper inspection
df_vertices_static = df_v_s.copy()
df_edges_static    = df_e_s.copy()
df_edges_static.insert(0, "sample_id", 0)

_out = config.GEOM_DATA_PATH
_out.mkdir(parents=True, exist_ok=True)

_vpath = _out / "static_grid_vertices.csv"
_epath = _out / "static_grid_edges.csv"

df_vertices_static.to_csv(_vpath, index=False)
df_edges_static.to_csv(_epath, index=False)

print(f"Saved: {_vpath}")
print(f"Saved: {_epath}")
print(f"  Vertices: {len(df_vertices_static)} rows  {list(df_vertices_static.columns)}")
print(f"  Edges:    {len(df_edges_static)} rows  {list(df_edges_static.columns)}")

### 6.3.1 Baseline comparison: static grid vs GA-optimised design
Runs the static regular grid through the **exact same pipeline** (feasibility -> cost matrix -> MILP -> GNN -> fitness) as the reference GA run, using identical stock, normalisation constants, fitness weights, and GNN model. The delta quantifies the material-outcome gain attributable to geometry adaptation alone.

In [ ]:
# =============================================================================
# Static grid vs GA_A — single-shot pipeline comparison
# =============================================================================
# Everything needed is loaded here so the cell is self-contained.
# The static grid uses params={} which triggers all-default vertex positions
# (top nodes on exact 3 m grid, bottom nodes at cell centres, no z-shift).
# =============================================================================

import json
import importlib
import pandas as pd
from pathlib import Path

import config
from workflows import c22_stage_geometry   as stage_geometry
from workflows import c27_stage_GNN        as stage_gnn
from workflows import c23_ga_evaluator     as ga_eval

for _m in [stage_geometry, stage_gnn, ga_eval]:
    importlib.reload(_m)

# ── Reference GA run ──────────────────────────────────────────────────────────
_GA_STEM = "GA_A_20260520_105730_RUN10_GEN250_EVAL7500_F-2_5262"

def _find_run_dir(stem: str) -> Path:
    direct = config.GA_DATA_PATH / stem
    if direct.is_dir():
        return direct
    for sub in config.GA_DATA_PATH.iterdir():
        if sub.is_dir():
            candidate = sub / stem
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError(f"Run directory not found for stem: {stem}")

_GA_DIR = _find_run_dir(_GA_STEM)

with open(_GA_DIR / f"{_GA_STEM}_run_config.json", encoding="utf-8") as f:
    _rc = json.load(f)
with open(_GA_DIR / f"{_GA_STEM}_best_design.json", encoding="utf-8") as f:
    _best = json.load(f)

_NORM    = _rc["normalization_constants"]   # C_max / R_max as used during the GA run
_MPREFIX = _rc["model_prefix"]
_GC      = _rc["ga_config"]

_GA_CONFIG_STATIC = {
    "fitness_weights":    _GC["fitness_weights"],
    "new_stock_max_uses": _GC["new_stock_max_uses"],
    "min_reuse_fraction": _GC["min_reuse_fraction"],
    "penalty_fitness":    _GC["penalty_fitness"],
    "w_structural_start": _GC["w_structural_start"],
    "w_structural_end":   _GC["w_structural_end"],
    "use_gnn":            _GC["use_gnn"],
}

print(f"GA run  : {_GA_STEM}")
print(f"Stock   : {_rc['stock']['source']}  ({_rc['stock']['n_total']} elements,  "
      f"NS={_rc['stock']['n_ns']}  RS={_rc['stock']['n_rs']})")
print(f"Norm    : C_max={_NORM['C_max']:.4f}   R_max={_NORM['R_max']:.4f}")
print(f"Weights : ω1={_GC['fitness_weights']['omega_1']}  "
      f"ω2={_GC['fitness_weights']['omega_2']}  "
      f"ω4={_GC['w_structural_end']}")

# ── Load stock A ──────────────────────────────────────────────────────────────
_stock_path = config.TIMBER_STOCK_PATH / "complete_timber_A.csv"
df_stock_A = None
for _opts in [{"sep": ";", "encoding": "utf-8"}, {"sep": ",", "encoding": "utf-8"},
              {"sep": ";", "encoding": "latin1"}, {"sep": ",", "encoding": "latin1"}]:
    try:
        _df = pd.read_csv(_stock_path, **_opts)
        if _df.shape[1] > 1:
            df_stock_A = _df
            break
    except Exception:
        pass
df_stock_A.columns = df_stock_A.columns.str.strip()
_prep_gnn_A = stage_gnn.prepare_stock_for_gnn(df_stock_A)
print(f"Stock A loaded: {len(df_stock_A)} elements\n")

# ── Run static grid through the full pipeline ────────────────────────────────
print("─" * 62)
print("Running static grid  (params={})  through full pipeline …")
print("─" * 62)

_static_eval = ga_eval.evaluate_design_candidate(
    design_params        = {},       # empty → all-default regular grid
    df_stock             = df_stock_A,
    fixed_norm_constants = _NORM,
    config_dict          = _GA_CONFIG_STATIC,
    model_prefix         = _MPREFIX,
    generation           = 250,      # end-of-run gen → w_structural = 0.8
    max_generations      = 250,
    sample_id            = 0,
    verbose              = True,
    prepared_gnn_stock   = _prep_gnn_A,
)

print(f"\nStatic eval  →  status: {_static_eval['status']}")
if _static_eval.get("reason"):
    print(f"  reason: {_static_eval['reason']}")

# ── Comparison table ──────────────────────────────────────────────────────────
_sfr = _static_eval.get("fitness_result", {})
_gfr = _best.get("fitness_result", {})

_metrics = [
    ("Fitness",                   _sfr.get("fitness",                  float("nan")), _best["fitness"]),
    ("Total cost  (kg CO₂e)",     _static_eval.get("total_cost",       float("nan")), _best["total_cost"]),
    ("Reuse fraction",            _static_eval.get("reuse_fraction",   float("nan")), _best["reuse_rate"]),
    ("GNN feasibility",           _static_eval.get("gnn_feasibility",  float("nan")), _best["gnn_feasibility"]),
    ("cost_norm",                 _sfr.get("cost_norm",                float("nan")), _gfr.get("cost_norm",  float("nan"))),
    ("reuse_norm",                _sfr.get("reuse_norm",               float("nan")), _gfr.get("reuse_norm", float("nan"))),
    ("structural infeasibility",  _sfr.get("structural_infeasibility", float("nan")), _gfr.get("structural_infeasibility", float("nan"))),
    ("MILP status",               _static_eval.get("milp_status", "—"),              _best["milp_status"]),
]

_rows = []
for label, sv, gv in _metrics:
    if isinstance(sv, float) and isinstance(gv, float):
        delta_str = f"{gv - sv:+.4f}"
    else:
        delta_str = "—"
    _rows.append({
        "Metric":           label,
        "Static grid":      f"{sv:.4f}" if isinstance(sv, float) else sv,
        f"GA best ({_GA_STEM[:4]})": f"{gv:.4f}" if isinstance(gv, float) else gv,
        "Δ  (GA − static)": delta_str,
    })

df_comparison = pd.DataFrame(_rows).set_index("Metric")

print(f"\n{'='*62}")
print(f"STATIC GRID  vs  GA-OPTIMISED  (stock A,  identical pipeline)")
print(f"{'='*62}")
display(df_comparison)

### 6.3.2 Cross-run comparison: GA_new / GA_A / GA_B
Full hard-metric comparison across stock scenarios. Normalisation bounds differ between runs and are excluded from the primary comparison, but listed in the summary for reference.

In [ ]:
# =============================================================================
# 6.5  Cross-run comparison — GA_new / GA_A / GA_B
# =============================================================================
# Edit RUN_STEMS to swap in different runs. Everything else is automatic.
# Hard values only — normalisation bounds differ across runs and are listed
# for reference only, not used in the primary comparison.
# =============================================================================

import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from pathlib import Path

import config
importlib.reload(config)

from workflows import c22_stage_geometry      as stage_geometry
from workflows import c24_stage_feasibility   as stage_feas
from workflows import c28_stage_fitness_score as stage_fitness

for _m in [stage_geometry, stage_feas, stage_fitness]:
    importlib.reload(_m)

# Pull the project colour palette once
_PC  = config.PLOT_COLORS
_EXT = _PC["extra_colors"]

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — set stems here
# ─────────────────────────────────────────────────────────────────────────────
RUN_STEMS = {
    "new": "GA_new_20260519_180633_GEN250_EVAL7500_F0_7394",
    "A":   "GA_A_20260519_192924_GEN250_EVAL7500_F-2_7279",
    "B":   "GA_B_20260520_132227_GEN250_EVAL7500_F-2_2732",
}

# Per-run colours (box plots / labels)
RUN_COLORS = {
    "new": _EXT["muted_teal"],
    "A":   _EXT["soft_sage_green"],
    "B":   _EXT["dusty_plum"],
}

# Member material colours
_C_RS  = _PC["RS"]           # #F2994B  reclaimed (orange)
_C_NS  = _PC["NS"]           # #61788C  new stock (blue)

# Node colours
_C_SUP  = _PC["upper_node"]  # #2F3E4F  supports (dark navy)
_C_LOAD = _PC["black"]       # #000000  top load nodes
_C_BOT  = _PC["lower_node"]  # #D9653B  bottom nodes

# ─────────────────────────────────────────────────────────────────────────────
# LOAD artefacts for each run
# ─────────────────────────────────────────────────────────────────────────────
def _resolve_run_dir(stem: str) -> Path:
    direct = config.GA_DATA_PATH / stem
    if direct.is_dir():
        return direct
    for sub in config.GA_DATA_PATH.iterdir():
        if sub.is_dir():
            candidate = sub / stem
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError(f"Run directory not found for stem: {stem}")

runs = {}
for label, stem in RUN_STEMS.items():
    d = _resolve_run_dir(stem)
    with open(d / f"{stem}_run_config.json",  encoding="utf-8") as f: rc = json.load(f)
    with open(d / f"{stem}_best_design.json", encoding="utf-8") as f: bd = json.load(f)
    tk = d / "top_k_designs"
    runs[label] = dict(
        stem       = stem,
        rc         = rc,
        bd         = bd,
        tk_summary = pd.read_csv(tk / f"{stem}_top10_summary.csv"),
        tk_verts   = pd.read_csv(tk / f"{stem}_top10_vertices.csv"),
        tk_edges   = pd.read_csv(tk / f"{stem}_top10_edges_assigned.csv"),
    )
    print(f"  {label:>3}  {stem}"
          f"\n       fitness={bd['fitness']:.4f}  reuse={bd['reuse_rate']:.1%}"
          f"  cost={bd['total_cost']:.1f} kg CO₂e\n")

# ─────────────────────────────────────────────────────────────────────────────
# WASTE CALCULATION — re-runs geometry + feasibility per best design
# ─────────────────────────────────────────────────────────────────────────────
def _load_stock_df(stock_source: str) -> pd.DataFrame:
    p = Path(stock_source)
    for opts in [{"sep": ";", "encoding": "utf-8"},
                 {"sep": ",", "encoding": "utf-8"},
                 {"sep": ";", "encoding": "latin1"},
                 {"sep": ",", "encoding": "latin1"}]:
        try:
            df = pd.read_csv(p, **opts)
            if df.shape[1] > 1:
                df.columns = df.columns.str.strip()
                return df
        except Exception:
            pass
    raise RuntimeError(f"Could not load stock from {p}")


def _compute_waste_for_run(r: dict) -> float:
    """Re-run geometry + feasibility for the best design, return offcut waste (m³)."""
    design_params = r["bd"].get("params", r["bd"].get("design_params", {}))
    df_stock      = _load_stock_df(r["rc"]["stock"]["source"])

    geo_out  = stage_geometry.run_geometry_from_design(design_params, sample_id=0)
    df_edges = geo_out["df_edges"]
    df_verts = geo_out["df_vertices"].copy()
    df_verts["v_idx"] = df_verts["vertex_index"].str.replace("v", "", regex=False).astype(int)
    df_verts = df_verts.sort_values("v_idx").reset_index(drop=True)

    node_positions = df_verts[["x", "y", "z"]].values
    support_nodes  = df_verts[df_verts["attribute"] == "support"]["v_idx"].tolist()
    load_nodes     = df_verts[df_verts["attribute"] == "load"]["v_idx"].tolist()

    df_slots, _, _, _ = stage_feas.build_cost_filter(
        node_positions = node_positions,
        edges_df       = df_edges,
        stock_df       = df_stock,
        support_nodes  = support_nodes,
        load_nodes     = load_nodes,
        verbose        = False,
    )

    milp_df = r["tk_edges"][r["tk_edges"]["rank"] == 1][["edge_id", "assigned_timber"]].copy()
    return stage_fitness.calculate_total_waste(milp_df, df_slots, df_stock)


print("Computing offcut waste for each run …")
for label in RUN_STEMS:
    try:
        waste = _compute_waste_for_run(runs[label])
        runs[label]["waste_m3"] = waste
        print(f"  GA_{label}: {waste:.4f} m³")
    except Exception as exc:
        runs[label]["waste_m3"] = float("nan")
        print(f"  GA_{label}: ERROR — {exc}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION A — Hard-metric summary table
# ─────────────────────────────────────────────────────────────────────────────
_metrics = [
    # objective
    ("Fitness (objective)",          lambda r: r["bd"]["fitness"]),
    # cost
    ("Total cost  (kg CO₂e)",        lambda r: r["bd"]["total_cost"]),
    ("Avg cost per member  (CO₂e)",  lambda r: r["bd"]["total_cost"] / 120),
    # reuse / material
    ("Reuse fraction",               lambda r: r["bd"]["reuse_rate"]),
    ("Reclaimed slots  (of 120)",    lambda r: round(r["bd"]["reuse_rate"] * 120)),
    # waste
    ("Offcut waste  (m³)",           lambda r: r["waste_m3"]),
    # structural
    ("GNN feasibility",              lambda r: r["bd"]["gnn_feasibility"]),
    ("Unsafe members  (GNN)",        lambda r: len(r["bd"]["gnn_unsafe_members"])),
    ("ω_structural applied",         lambda r: r["bd"]["w_structural"]),
    # optimisation run
    ("Best found — generation",      lambda r: r["bd"]["generation"]),
    ("MILP status",                  lambda r: r["bd"]["milp_status"]),
    ("GA evaluations",               lambda r: r["rc"]["n_evals"]),
    # stock pool
    ("Stock — total elements",       lambda r: r["rc"]["stock"]["n_total"]),
    ("Stock — NS",                   lambda r: r["rc"]["stock"]["n_ns"]),
    ("Stock — RS",                   lambda r: r["rc"]["stock"]["n_rs"]),
    ("RS availability  (%)",         lambda r: r["rc"]["stock"]["n_rs"] / r["rc"]["stock"]["n_total"]),
    # norm bounds — reference only
    ("C_max  [norm ref]",            lambda r: r["rc"]["normalization_constants"]["C_max"]),
    ("R_max  [norm ref]",            lambda r: r["rc"]["normalization_constants"]["R_max"]),
]

tbl = {}
for label in RUN_STEMS:
    col = {}
    for name, fn in _metrics:
        try:   col[name] = fn(runs[label])
        except Exception: col[name] = "—"
    tbl[f"GA_{label}"] = col

df_tbl = pd.DataFrame(tbl).rename_axis("Metric")

def _fmt(v):
    if isinstance(v, float): return f"{v:.4f}"
    if isinstance(v, int):   return str(v)
    return v

df_display = df_tbl.apply(lambda col: col.map(_fmt))

print(f"\n{'='*58}\nHARD-METRIC SUMMARY\n{'='*58}")
display(df_display)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION B — Top-10 distribution box plots
# ─────────────────────────────────────────────────────────────────────────────
_box_defs = [
    ("fitness",         "Fitness",              None),
    ("total_cost",      "Total cost (kg CO₂e)", None),
    ("reuse_rate",      "Reuse fraction",        (0, 1)),
    ("gnn_feasibility", "GNN feasibility",       (0, 1)),
]

fig_box, axes_b = plt.subplots(1, 4, figsize=(16, 4.5))
fig_box.suptitle("Top-10 solution distributions  (GA_new / GA_A / GA_B)",
                 fontweight="bold", fontsize=11)

labels_list = list(RUN_STEMS.keys())
colors_list = [RUN_COLORS[l] for l in labels_list]

for ax, (col, title, ylim) in zip(axes_b, _box_defs):
    data  = [runs[l]["tk_summary"][col].dropna().values for l in labels_list]
    bp = ax.boxplot(data, patch_artist=True, widths=0.45,
                    medianprops=dict(color=_PC["black"], lw=2))
    for patch, color in zip(bp["boxes"], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
    for i, (d, color) in enumerate(zip(data, colors_list), start=1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.08, 0.08, len(d)),
                   d, color=color, s=22, alpha=0.88, zorder=5)
    ax.set_xticks(range(1, len(labels_list) + 1))
    ax.set_xticklabels([f"GA_{l}" for l in labels_list], fontsize=9)
    ax.set_title(title, fontsize=9.5, fontweight="bold")
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
    if ylim:
        ax.set_ylim(*ylim)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION C — Best-design geometry: perspective + top view, RS/NS coloured
# ─────────────────────────────────────────────────────────────────────────────
n_runs = len(RUN_STEMS)

fig_geo, axes_geo = plt.subplots(
    2, n_runs,
    figsize=(6.0 * n_runs, 11),
    subplot_kw={"projection": "3d"},
)

fig_geo.suptitle(
    "Best-design geometry — orange = reclaimed RS  ·  blue = new NS",
    fontweight="bold", fontsize=11,
)

_VIEWS = [
    (26, -50, "Perspective"),
    (88, -90, "Top view"),
]

for col_idx, label in enumerate(RUN_STEMS):
    r   = runs[label]
    bd  = r["bd"]

    vdf = r["tk_verts"][r["tk_verts"]["rank"] == 1].copy()
    vdf["vi"] = vdf["vertex_index"].str[1:].astype(int)
    vlookup = vdf.set_index("vi")[["x", "y", "z", "layer", "attribute"]]

    edf = r["tk_edges"][r["tk_edges"]["rank"] == 1].copy()
    rs_count = edf["assigned_timber"].astype(str).str.startswith("RS").sum()

    # ── Pre-build edge segment arrays (RS and NS) ──────────────────────────
    v1s = edf["V1"].astype(int).values
    v2s = edf["V2"].astype(int).values
    valid = np.isin(v1s, vlookup.index) & np.isin(v2s, vlookup.index)
    v1s, v2s = v1s[valid], v2s[valid]
    is_rs_arr = edf["assigned_timber"].astype(str).str.startswith("RS").values[valid]

    p1_coords = vlookup.loc[v1s, ["x", "y", "z"]].values
    p2_coords = vlookup.loc[v2s, ["x", "y", "z"]].values
    segs = np.stack([p1_coords, p2_coords], axis=1)  # (n_valid, 2, 3)

    # ── Node masks (compute once per run) ─────────────────────────────────
    _sup_v = vdf["attribute"] == "support"
    _bot_v = (~_sup_v) & (vdf["layer"] == "bottom")
    _top_v = ~_sup_v & ~_bot_v
    _vxyz  = vdf[["x", "y", "z"]].values

    for row_idx, (elev, azim, view_lbl) in enumerate(_VIEWS):
        ax = axes_geo[row_idx][col_idx]

        for is_rs, col in [(True, _C_RS), (False, _C_NS)]:
            mask = is_rs_arr == is_rs
            if mask.any():
                ax.add_collection3d(Line3DCollection(segs[mask], color=col, lw=1.5, alpha=1.0))

        for mask, col, ms in [(_sup_v, _C_SUP, 55), (_bot_v, _C_BOT, 20), (_top_v, _C_LOAD, 20)]:
            if mask.any():
                ax.scatter3D(_vxyz[mask, 0], _vxyz[mask, 1], _vxyz[mask, 2],
                             c=col, s=ms, zorder=5)

        ax.auto_scale_xyz(vdf["x"].values, vdf["y"].values, vdf["z"].values)
        ax.set_xlabel("x [m]", fontsize=7)
        ax.set_ylabel("y [m]", fontsize=7)
        ax.set_zlabel("z [m]" if azim != -90 else "", fontsize=7)
        ax.set_box_aspect((1, 0.65, 0.28))
        ax.view_init(elev=elev, azim=azim)
        ax.tick_params(labelsize=6)

        if row_idx == 0:
            ax.set_title(
                f"GA_{label}  ·  {view_lbl}\n"
                f"reuse={bd['reuse_rate']:.1%}  ({rs_count}/120 RS)  ·  "
                f"cost={bd['total_cost']:.1f}  ·  fit={bd['fitness']:.4f}",
                fontsize=8.5, fontweight="bold", pad=8,
            )
        else:
            ax.set_title(view_lbl, fontsize=8, pad=4)

_leg = [
    mpatches.Patch(color=_C_RS,   label="Reclaimed (RS)"),
    mpatches.Patch(color=_C_NS,   label="New stock (NS)"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_SUP,  markersize=8, label="Support"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_LOAD, markersize=6, label="Top load node"),
    plt.Line2D([], [], marker="o", color="w", markerfacecolor=_C_BOT,  markersize=6, label="Bottom node"),
]
axes_geo[0][0].legend(handles=_leg, loc="upper left", fontsize=7.5,
                      framealpha=0.9, bbox_to_anchor=(-0.06, 1.22))

plt.tight_layout()
plt.show()

### 6.2.3 New stock max use comparison
Comparison of 3 runs of new stock using max stock 10 items and 3 runs of new stock using max stock items len(slot elements) aka 120. This is to anilyse if a GA optimizer will still go near a geometric optimum or a human designers perspective

## 6.3 Design outcome

This section presents the design outcome of the best-performing GA_A run. It moves from the raw geometry and material composition through structural validation to an architectural reading of the form — bridging the computational result to the building technology argument.

### 6.3.1 Geometry

The optimised geometry departs from the regular 3 m grid baseline in all 15 bottom node positions, identified by the CMA-ES as configurations that improve reuse fraction — from 30.3% on the static grid to [**X%**] on the best design. Figure [X] below colours each of the 120 members by material provenance: **orange** = reclaimed stock (RS), **blue** = new stock (NS). Reclaimed members concentrate in the top chord and longer diagonal web members, where the existing RS element lengths are most compatible with the force distribution. A top-view projection (right) confirms the near-rectilinear planning footprint is preserved despite the bottom-layer irregularity.

> **[Figure 6.3.1]** — Generated by the code cell below. Export as SVG/PDF for thesis (200 dpi minimum for print).

In [ ]:
# =============================================================================
# 6.3.1  Optimised geometry — RS / NS member colouring
# =============================================================================
# Reference run: set GA_STEM to the final best-run stem before submitting.
# =============================================================================

import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from pathlib import Path

import config
importlib.reload(config)

_PC  = config.PLOT_COLORS
_C_RS   = _PC["RS"]           # #F2994B  reclaimed (orange)
_C_NS   = _PC["NS"]           # #61788C  new stock (blue)
_C_SUP  = _PC["upper_node"]   # #2F3E4F  supports
_C_LOAD = _PC["black"]        # #000000  top load nodes
_C_BOT  = _PC["lower_node"]   # #D9653B  bottom nodes

# ── Reference run — update to final run stem before submitting ────────────────
GA_STEM = "GA_A_20260519_192924_GEN250_EVAL7500_F-2_7279"
GA_DIR  = config.GA_DATA_PATH / GA_STEM
TK_DIR  = GA_DIR / "top_k_designs"

with open(GA_DIR / f"{GA_STEM}_best_design.json", encoding="utf-8") as f:
    bd = json.load(f)

vdf = pd.read_csv(TK_DIR / f"{GA_STEM}_top10_vertices.csv")
edf = pd.read_csv(TK_DIR / f"{GA_STEM}_top10_edges_assigned.csv")

vdf = vdf[vdf["rank"] == 1].copy()
edf = edf[edf["rank"] == 1].copy()

vdf["vi"] = vdf["vertex_index"].str[1:].astype(int)
vlookup   = vdf.set_index("vi")[["x", "y", "z", "layer", "attribute"]]

v1s    = edf["V1"].astype(int).values
v2s    = edf["V2"].astype(int).values
valid  = np.isin(v1s, vlookup.index) & np.isin(v2s, vlookup.index)
is_rs  = edf["assigned_timber"].astype(str).str.startswith("RS").values[valid]

p1c = vlookup.loc[v1s[valid], ["x","y","z"]].values
p2c = vlookup.loc[v2s[valid], ["x","y","z"]].values
segs = np.stack([p1c, p2c], axis=1)

rs_count = is_rs.sum()
ns_count = (~is_rs).sum()

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": "3d"})
fig.suptitle(
    f"GA_A best design  —  {rs_count} RS (orange)  ·  {ns_count} NS (blue)  "
    f"·  reuse {bd['reuse_rate']:.1%}  ·  cost {bd['total_cost']:.1f} kg CO₂e  "
    f"·  fitness {bd['fitness']:.4f}",
    fontweight="bold", fontsize=10.5,
)

for ax, (elev, azim, title) in zip(axes, [(25, -50, "Perspective"), (88, -90, "Top view")]):
    for is_r, col, lw, alpha in [(True, _C_RS, 2.2, 0.95), (False, _C_NS, 1.0, 0.50)]:
        m = is_rs == is_r
        if m.any():
            ax.add_collection3d(Line3DCollection(segs[m], color=col, lw=lw, alpha=alpha))

    _sup = vdf["attribute"] == "support"
    _bot = (~_sup) & (vdf["layer"] == "bottom")
    _top = ~_sup & ~_bot
    xyz  = vdf[["x","y","z"]].values
    for mask, col, ms in [(_sup, _C_SUP, 60), (_bot, _C_BOT, 24), (_top, _C_LOAD, 18)]:
        if mask.any():
            ax.scatter3D(xyz[mask,0], xyz[mask,1], xyz[mask,2], c=col, s=ms, zorder=5)

    ax.auto_scale_xyz(vdf["x"].values, vdf["y"].values, vdf["z"].values)
    ax.set_xlabel("x [m]", fontsize=8); ax.set_ylabel("y [m]", fontsize=8)
    ax.set_zlabel("z [m]" if elev < 80 else "", fontsize=8)
    ax.set_box_aspect((1, 0.65, 0.28))
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.tick_params(labelsize=7)

axes[0].legend(handles=[
    mpatches.Patch(color=_C_RS,   label=f"Reclaimed  (RS) — {rs_count} members"),
    mpatches.Patch(color=_C_NS,   label=f"New stock  (NS) — {ns_count} members"),
    plt.Line2D([],[],marker="o",color="w",markerfacecolor=_C_SUP, ms=8, label="Support node"),
    plt.Line2D([],[],marker="o",color="w",markerfacecolor=_C_LOAD,ms=6, label="Top load node"),
    plt.Line2D([],[],marker="o",color="w",markerfacecolor=_C_BOT, ms=6, label="Bottom node"),
], loc="upper left", fontsize=8, framealpha=0.9, bbox_to_anchor=(-0.05, 1.12))

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

### 6.3.2 Bill of materials

Table [X] lists the complete 120-member assignment for the best design, grouped by material state. Each reclaimed element's embodied carbon is limited to end-of-life cutting, transport, and re-processing (LCA phases C1–C3, A5), while new-stock elements carry the full production burden (A1-A3). The assignment therefore concentrates the highest-carbon operations on structural positions where NS is unavoidable, and reserves RS for the larger-volume, more demanding slots.

> **[Table 6.3.2a]** — Aggregate bill of materials: RS vs NS count, volume, and CO₂e  
> **[Table 6.3.2b]** — Cross-section inventory: unique Width × Depth combinations used  
> **[Figure 6.3.2]** — CO₂ penalty distribution: box plot per member type (top chord / web / bottom chord)

In [ ]:
# =============================================================================
# 6.3.2  Bill of materials
# =============================================================================

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import config

_PC  = config.PLOT_COLORS

GA_STEM = "GA_A_20260519_192924_GEN250_EVAL7500_F-2_7279"
GA_DIR  = config.GA_DATA_PATH / GA_STEM
TK_DIR  = GA_DIR / "top_k_designs"

# ── Load rank-1 edges + stock CSV ─────────────────────────────────────────────
edf = pd.read_csv(TK_DIR / f"{GA_STEM}_top10_edges_assigned.csv")
edf = edf[edf["rank"] == 1].copy()

df_stock = pd.read_csv(GA_DIR / f"{GA_STEM}_stock.csv", sep=";")
df_stock.columns = df_stock.columns.str.strip()

# ── Load rank-1 vertices for member-type classification ───────────────────────
vdf = pd.read_csv(TK_DIR / f"{GA_STEM}_top10_vertices.csv")
vdf = vdf[vdf["rank"] == 1].copy()
vlookup = vdf.set_index("vertex_index")[["layer"]].to_dict("index")

def _vk(v):
    s = str(v)
    return s if s.startswith("v") else f"v{s}"

def _member_type(row):
    l1 = vlookup.get(_vk(row["V1"]), {}).get("layer", "")
    l2 = vlookup.get(_vk(row["V2"]), {}).get("layer", "")
    if l1 == l2 == "top":    return "Top chord"
    if l1 == l2 == "bottom": return "Bottom chord"
    return "Web / diagonal"

edf["member_type"] = edf.apply(_member_type, axis=1)

# ── Join with stock ───────────────────────────────────────────────────────────
merged = edf.merge(
    df_stock[["Member_ID", "State", "Length", "Width", "Depth", "EmissionFactor", "Transport_Dist"]],
    left_on="assigned_timber", right_on="Member_ID", how="left",
)
merged["material"] = merged["assigned_timber"].str[:2]       # RS or NS
merged["Length_m"]  = merged["Length"]  / 1000
merged["Width_m"]   = merged["Width"]   / 1000
merged["Depth_m"]   = merged["Depth"]   / 1000
merged["Volume_m3"] = merged["Length_m"] * merged["Width_m"] * merged["Depth_m"]

# ── Aggregate summary ─────────────────────────────────────────────────────────
summary = (
    merged.groupby("material")
    .agg(
        Count       = ("edge_id",    "count"),
        Volume_m3   = ("Volume_m3",  "sum"),
        CO2e        = ("CO2_Penalty","sum"),
        Avg_CO2e    = ("CO2_Penalty","mean"),
        Avg_Length  = ("Length_m",   "mean"),
    )
    .round(3)
)
summary.loc["Total"] = summary.sum(numeric_only=True)
summary.loc["Total", "Avg_CO2e"]   = merged["CO2_Penalty"].mean()
summary.loc["Total", "Avg_Length"] = merged["Length_m"].mean()

print("=" * 55)
print("BILL OF MATERIALS — aggregate by material state")
print("=" * 55)
display(summary.rename_axis("Material"))

# ── Cross-section inventory ───────────────────────────────────────────────────
xs_inv = (
    merged.groupby(["material", "Width", "Depth"])
    .agg(Count=("edge_id","count"), Total_length_m=("Length_m","sum"))
    .reset_index()
    .sort_values(["material","Count"], ascending=[True, False])
)
xs_inv.insert(2, "Section", xs_inv["Width"].astype(int).astype(str) + "×" + xs_inv["Depth"].astype(int).astype(str) + " mm")
xs_inv = xs_inv.drop(columns=["Width","Depth"])

print("\n" + "=" * 55)
print("CROSS-SECTION INVENTORY")
print("=" * 55)
display(xs_inv.set_index(["material","Section"]))

# ── Figure: CO2 distribution by member type ───────────────────────────────────
mt_order = ["Top chord", "Web / diagonal", "Bottom chord"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("CO₂e penalty distribution by member position", fontweight="bold", fontsize=11)

# Left: box plot per member type, split RS/NS
for ax, mat, col in zip(
    [axes[0], axes[1]],
    ["RS", "NS"],
    [_PC["RS"], _PC["NS"]],
):
    sub = merged[merged["material"] == mat]
    data = [sub[sub["member_type"] == mt]["CO2_Penalty"].values for mt in mt_order]
    bp = ax.boxplot(data, patch_artist=True, widths=0.45,
                    medianprops=dict(color=_PC["black"], lw=2))
    for patch in bp["boxes"]:
        patch.set_facecolor(col); patch.set_alpha(0.72)
    for i, d in enumerate(data, 1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.07, 0.07, len(d)),
                   d, color=col, s=20, alpha=0.85, zorder=5)
    ax.set_xticks([1,2,3]); ax.set_xticklabels(mt_order, fontsize=8.5)
    ax.set_ylabel("CO₂e penalty (kg)", fontsize=9)
    ax.set_title(f"{'Reclaimed (RS)' if mat=='RS' else 'New stock (NS)'} — {(merged['material']==mat).sum()} members",
                 fontsize=10, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=config.PLOT_STYLE["grid_alpha"])
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

### 6.3.3 Karamba structural validation

The MILP-assigned geometry and member cross-sections were re-imported into Karamba3D in Grasshopper for finite-element verification. The utilisation ratio (UC = applied force / section capacity) was computed for each member under the same load case used in training data generation (uniformly distributed gravity load applied at all top nodes).

> **[Figure 6.3.3a]** — Karamba3D member utilisation plot. Colour scale: blue (UC ≈ 0) → green (UC ≈ 0.5) → red (UC ≥ 1.0). Export from Rhino Raytraced or Grasshopper `Karamba3D > BeamView` component. Members shown in red are overstressed (UC > 1.0) and correspond to the GNN-flagged unsafe members.
>
> **[Figure 6.3.3b]** *(optional)* — Exploded axonometric showing top chord, bottom chord, and web members separately, each labelled with RS / NS count.

The Karamba3D result confirms [**X**] members with UC > 1.0, consistent with the GNN's prediction of [**X**] unsafe members (prediction error: [**X**] members, [**X%**] of the truss). The [**X**] overstressed members are predominantly [*top chord / web / diagonal*] elements, where [*describe pattern — e.g., the reclaimed sections selected were slightly undersized for peak compression loads*]. This is an expected outcome of the GNN operating as a proxy rather than a solver: it penalises predicted-unsafe geometry during optimisation but does not guarantee full structural compliance. A post-optimisation section-upgrade step (replacing flagged members with the next-larger feasible stock item) would eliminate all UC violations without re-running the GA.

### 6.3.4 Architectural reading

The stock-informed space frame occupies a [15 m × 9 m] plan footprint and spans [1.5 m] in structural depth. In plan the top chord traces a near-rectilinear grid — readable as a conventional structural grid from above — while the bottom chord warps: the 15 bottom nodes are displaced laterally and vertically from the regular-grid default by the CMA-ES search. This warp is not aesthetic intent but material logic: each bottom-node position was shifted to bring the required member lengths closer to the available RS element lengths, reducing the need for cutting and the total embodied carbon.

> **[Figure 6.3.4a]** — Perspective render from Rhino Raytraced mode or Enscape, viewpoint from below looking toward one of the short edges. Orange members (RS) should be rendered with a warm timber tone; blue members (NS) with a fresh-cut pale tone. Structural node connectors (steel plates or bolted joints) can be added as schematic geometry.
>
> **[Figure 6.3.4b]** — Floor-plan projection (top view) with member provenance as a heatmap overlay: opacity encodes the ratio of RS to total assignment per grid cell.
>
> **[Figure 6.3.4c]** *(optional)* — Section cut through the short axis showing the structural depth and the asymmetric bottom-chord position relative to the support corners.

From below, the geometry reads as a shallow vault: the irregular node positions create a sense of depth variation across the span, despite the constant overall depth. This spatial effect is produced directly by the stock composition — in a different RS inventory, the GA would produce a different warp, a different spatial character. The form is therefore not fixed but *stock-contingent*: a family of geometries parametrised by supply chain, each structurally valid, each materially unique. This positions the design system within a broader trajectory of *data-driven adaptive reuse*: buildings whose form is authored not by the designer's geometric preference but by the residual geometry of the material world.